# NestedSimPy — Movie Renege

[NestedSimPy](https://nestedsimpy.github.io/) adapts SimPy's official [Movie Renege example](https://simpy.readthedocs.io/en/latest/examples/movie_renege.html) — moviegoers queueing for tickets and leaving when a film sells out — so the outer behavior is unchanged while inner branches are launched at each trigger event. This example uses `NestedResource`.

See the [example page](https://nestedsimpy.github.io/official-parity/movie-reneging.html) for the side-by-side plain/nested code.

## 1. Install

_Pre-release: NestedSimPy installs from a hosted wheel. (After the public release this becomes `pip install nestedsimpy`.)_

In [ ]:
# Pre-release install from a hosted wheel (Google Drive).
!pip install -q gdown
import gdown
gdown.download(id="1N7mlgDVpVids6Ekr4p2e-gEUrUodiEuq",
               output="nestedsimpy-0.1.0-py3-none-any.whl", quiet=True)
!pip install -q "nestedsimpy-0.1.0-py3-none-any.whl[plot]"

import nestedsimpy
print("NestedSimPy ready —", len(nestedsimpy.__all__), "public objects")

## 2. Run the nested example

The model is written to a file and run as a subprocess; the output below is the **outer** trajectory and matches the plain SimPy example.

In [ ]:
%%writefile movie_reneging_colab.py
from __future__ import annotations

# --- inline prelude (replaces the examples' local _imports shim) ---
import argparse, os, random, shutil, sys, itertools
from pathlib import Path

import simpy
import nestedsimpy
from nestedsimpy import (
    NestedEnvironment, NestedResource, NestedPreemptiveResource,
    NestedStore, NestedContainer,
)
try:
    from nestedsimpy.postprocess import (
        package_latest_run, relocate_raw_artifacts, export_realizations,
    )
except Exception:  # pragma: no cover
    package_latest_run = relocate_raw_artifacts = export_realizations = None

DEFAULT_OUT_ROOT = Path("nested_output")
DEFAULT_AUTOPLOT = False
REPO_ROOT = Path(".")
PACKAGE_ROOT = Path(".")

def default_out(*parts):
    p = DEFAULT_OUT_ROOT.joinpath(*map(str, parts)); p.mkdir(parents=True, exist_ok=True); return p

def set_nested_output_folder(*parts):
    p = Path(os.path.join(*[str(x) for x in parts])); p.mkdir(parents=True, exist_ok=True); return p
# --- end prelude ---

"""
Movie renege example (nested-sim adaptation).

Covers:

- Resources: Resource
- Condition events
- Shared events

Scenario:
  A movie theatre has one ticket counter selling tickets for three
  movies (next show only). When a movie is sold out, all people waiting
  to buy tickets for that movie renege (leave queue).
"""


import itertools
import random
from typing import Dict, List, NamedTuple, Optional


# fmt: off
RANDOM_SEED = 42
TICKETS = 50          # Number of tickets per movie (matches plain example)
SELLOUT_THRESHOLD = 2  # Fewer tickets than this is a sellout
SIM_TIME = 120       # Simulate until
# fmt: on

NESTED_OUTPUT_FOLDER = set_nested_output_folder("simpy_examples", "movie_reneging")


def moviegoer(env, cust_id, movie, num_tickets, theater):
    """A moviegoer tries to by a number of tickets (*num_tickets*) for
    a certain *movie* in a *theater*.

    If the movie becomes sold out, she leaves the theater. If she gets
    to the counter, she tries to buy a number of tickets. If not enough
    tickets are left, she argues with the teller and leaves.

    If at most one ticket is left after the moviegoer bought her
    tickets, the *sold out* event for this movie is triggered causing
    all remaining moviegoers to leave.

    """

    with theater.counter.request(job_id=cust_id) as my_turn:
        result = yield my_turn | theater.sold_out[movie]
        if my_turn not in result:
            theater.num_renegers[movie] += 1
            return

        if theater.available[movie] < num_tickets:
            yield env.nested_timeout(
                {"distribution": "deterministic", "value": 0.5},
                label="argue_then_leave",
            )
            return

        theater.available[movie] -= num_tickets
        if theater.available[movie] < SELLOUT_THRESHOLD:
            theater.sold_out[movie].succeed()
            theater.when_sold_out[movie] = env.now
            theater.available[movie] = 0
        yield env.nested_timeout(
            {"distribution": "deterministic", "value": 1.0}, label="purchase"
        )


def customer_arrivals(env, theater):
    """Create new *moviegoers* until the sim time reaches 120."""

    cust_seq = itertools.count()
    while True:
        yield env.nested_timeout(
            {"distribution": "exponential", "lambda": 1 / 0.5}, label="interarrival"
        )
        cust_id = next(cust_seq)
        movie = random.choice(theater.movies)
        num_tickets = random.randint(1, 6)
        if theater.available[movie]:
            env.process(moviegoer(env, cust_id, movie, num_tickets, theater))


class Theater(NamedTuple):
    """Container bundling the shared state for the movie-theater example."""

    counter: NestedResource
    movies: List[str]
    available: Dict[str, int]
    sold_out: Dict[str, simpy.Event]
    when_sold_out: Dict[str, Optional[float]]
    num_renegers: Dict[str, int]


def main():
    """Run the nested-sim movie renege example and package the outputs."""

    random.seed(RANDOM_SEED)
    env = NestedEnvironment()
    env._ns_print_branch_summary = False

    # Output options first.
    env.set_output_options(out_dir=str(NESTED_OUTPUT_FOLDER), gzip_trace=False)
    env.set_post_processing_options(
        gzip_trace=False,
        package_latest=True,
        print_outputs=False,
        autoplot=DEFAULT_AUTOPLOT,
        autoplot_example="movie_reneging",
    )
    # Outer settings.
    env.set_rng("independent")
    env.set_outer_seed(RANDOM_SEED)
    env.set_triggering_objects(nested_id="counter")
    env.set_triggering_conditions({"on": "arrival", "frequency": 1})
    env.set_outer_stopping_condition(timeout=SIM_TIME)
    # Inner settings.
    env.set_inner_repetitions(1)
    env.set_inner_stopping_condition(
        relative_time=10.0, triggering_customer_departs=True
    )

    movies = ["Python Unchained", "Kill Process", "Pulp Implementation"]
    theater = Theater(
        counter=NestedResource(env, capacity=1, nested_id="counter", snapshot=False),
        movies=movies,
        available={movie: TICKETS for movie in movies},
        sold_out={movie: env.event() for movie in movies},
        when_sold_out={movie: None for movie in movies},
        num_renegers={movie: 0 for movie in movies},
    )

    env.process(customer_arrivals(env, theater))
    env.nested_run()

    for movie in movies:
        if theater.sold_out[movie]:
            sellout_time = theater.when_sold_out[movie]
            num_renegers = theater.num_renegers[movie]
            print(
                f'Movie "{movie}" sold out {sellout_time:.1f} minutes '
                f"after ticket counter opening."
            )
            print(
                f"  Number of people leaving queue when film sold out: {num_renegers}"
            )


if __name__ == "__main__":
    main()


In [ ]:
# Run as a subprocess so the outer output is clean (inner branches run in separate processes).
!python movie_reneging_colab.py

## 3. Inspect the run

`OutputManager` reads the run folder and reports the trigger events, plots the outer trajectory, and exports the sample path.

In [ ]:
import glob, os
run = os.path.dirname(glob.glob("simpy_examples/movie_reneging/**/raw", recursive=True)[0])

from nestedsimpy import OutputManager
om = OutputManager(run)
print(f"{len(om.triggers())} trigger events; the outer path has {len(om.export_outer_event_log())} recorded events")

om.visualize_outer_static("outer.png")   # outer trajectory, saved as a static image
fig = om.visualize_outer_interactive()   # the same trajectory as an interactive plot
fig.show()

om.export_outer_event_log("outer.csv")   # the outer event log as a CSV
print("wrote outer.csv")